# Tracking the Portfolio During the Competition

A check-in for the portfolio in `results/allocation.csv`, which `1_build_portfolio.ipynb` creates.

**How it works.** The portfolio is *buy and hold*: the number of shares is fixed on the purchase day (Sep 14, 2026) and never changes. As prices move, each stock's share of the portfolio drifts. This notebook values those fixed shares at the latest closing prices and compares the result with two references:
- **Equal weight:** the same 10 stocks at 10% each, which shows whether the chosen *weights* helped
- **SPY:** the S&P 500, which shows how the whole market did

**When to run it.** Any day after the market closes (4 p.m. New York time), for example once a week. It downloads fresh prices every time.

**Before the first run.** If the actual purchase prices differed from the Sep 14 closing prices, enter them in `FILL_PRICES` below.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import FuncFormatter, PercentFormatter

from stock_competition import charts, data
from stock_competition.market_calendar import trading_days
from stock_competition.paths import RESULTS_DIR

%config InlineBackend.figure_format = "retina"
charts.set_chart_style()

In [ ]:
ALLOCATION_FILE = RESULTS_DIR / "allocation.csv"
START, END = "2026-09-14", "2026-12-14"   # buy at the START close; final value at the END close
BENCHMARK = "SPY"
FILL_PRICES = {}        # actual purchase prices if they differ from the START close, e.g. {"NVDA": 218.50}
REFRESH_DATA = True     # download the latest prices on every run

In [ ]:
allocation = pd.read_csv(ALLOCATION_FILE)
tickers = allocation["ticker"].tolist()
capital = allocation["dollars"].sum()

prices = data.load_prices(tickers + [BENCHMARK], refresh=REFRESH_DATA).ffill()
held = prices.loc[(prices.index >= pd.Timestamp(START)) & (prices.index <= pd.Timestamp(END))]
assert len(held), f"No closing prices on or after {START} yet. Run this after the market closes on {START}."

buy_date = held.index[0]
buy_price = held.iloc[0].copy()
for ticker, price in FILL_PRICES.items():
    buy_price[ticker] = price
shares = allocation.set_index("ticker")["dollars"] / buy_price[tickers]

values = held[tickers] * shares
portfolio = values.sum(axis=1)
equal_weight = (held[tickers] / buy_price[tickers]).mean(axis=1) * capital
benchmark = held[BENCHMARK] / buy_price[BENCHMARK] * capital

last_date = held.index[-1]
day = len(trading_days(buy_date, last_date))
total_days = len(trading_days(START, END))
print(f"Bought {buy_date:%Y-%m-%d} · latest close {last_date:%Y-%m-%d} · trading day {day} of {total_days}")

summary = pd.DataFrame({
    "value": [portfolio.iloc[-1], equal_weight.iloc[-1], benchmark.iloc[-1]],
    "return": [portfolio.iloc[-1] / capital - 1, equal_weight.iloc[-1] / capital - 1,
               benchmark.iloc[-1] / capital - 1],
}, index=["Our portfolio", "Equal weight (same stocks)", BENCHMARK])
display(summary.style.format({"value": "${:,.0f}", "return": "{:+.2%}"}))

## Positions
- `weight_at_buy`: the planned weight on the purchase day
- `weight_now`: the stock's share of the portfolio today, after prices moved
- `contribution`: how many percentage points the stock added to (or took from) the portfolio's return, equal to its buy weight × its own return. The contributions add up to the portfolio's return.

In [ ]:
positions = pd.DataFrame({
    "weight_at_buy": allocation.set_index("ticker")["weight"],
    "shares": shares,
    "buy_price": buy_price[tickers],
    "last_price": held[tickers].iloc[-1],
    "value": values.iloc[-1],
})
positions["return"] = positions["last_price"] / positions["buy_price"] - 1
positions["contribution"] = positions["value"] / capital - positions["weight_at_buy"]
positions["weight_now"] = positions["value"] / positions["value"].sum()
positions = positions.sort_values("contribution", ascending=False)
assert np.isclose(positions["contribution"].sum(), portfolio.iloc[-1] / capital - 1)

display(positions.style.format({
    "weight_at_buy": "{:.1%}", "shares": "{:,.4f}", "buy_price": "${:,.2f}", "last_price": "${:,.2f}",
    "value": "${:,.0f}", "return": "{:+.2%}", "contribution": "{:+.2%}", "weight_now": "{:.1%}",
}))

## Charts
**Return since purchase:** each line starts at 0% on the purchase day. The x-axis runs to the end of the competition, so the empty space on the right is the time left.

**Contribution:** blue bars added to the portfolio's return, red bars subtracted from it.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for series, label, color in [(portfolio, "Our portfolio", charts.COLORS["blue"]),
                             (equal_weight, "Equal weight", charts.COLORS["orange"]),
                             (benchmark, BENCHMARK, charts.COLORS["aqua"])]:
    ax.plot(series.index, series / capital - 1, color=color, label=label, marker="o" if len(series) == 1 else None)
latest_return = portfolio.iloc[-1] / capital - 1
ax.annotate(f"{latest_return:+.1%}", (last_date, latest_return), xytext=(8, 0), textcoords="offset points",
            va="center", fontsize=9, color=charts.INK["primary"])
ax.axhline(0, color=charts.INK["axis"], linewidth=0.8)
ax.set_xlim(float(mdates.date2num(buy_date)), float(mdates.date2num(pd.Timestamp(END))))
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4, maxticks=8))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.legend(loc="lower right", bbox_to_anchor=(1, 1), ncols=3)
ax.set_title("Return since purchase")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.2))
contribution = positions.sort_values("contribution")["contribution"]
ax.barh(contribution.index, contribution, height=0.7,
        color=np.where(contribution >= 0, charts.COLORS["blue"], charts.COLORS["red"]))
ax.axvline(0, color=charts.INK["axis"], linewidth=0.8)
ax.grid(axis="y", visible=False)
ax.grid(axis="x", visible=True)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.2%}"))
ax.set_title("Contribution to portfolio return (blue adds, red subtracts)")
plt.show()